In [188]:
import numpy as np
import sklearn
import torch
import os

In [189]:
if not os.path.exists('tree_species_classifier_data.npz'):
  !wget -O tree_species_classifier_data.npz "https://www.dropbox.com/scl/fi/b7mw23k3ifaeui9m8nnn3/tree_species_classifier_data.npz?rlkey=bgxp37c1t04i7q35waf3slc26&dl=1"

In [190]:
data = np.load('tree_species_classifier_data.npz')
train_features = data['train_features']
train_labels = data['train_labels']
test_features = data['test_features']
test_labels = data['test_labels']

### Data Exploration

In [191]:
matrices = [train_features, train_labels, test_features, test_labels]
for matrix in matrices:
    print("Type:", type(matrix))
    print("Shape:", matrix.shape)
    print("Range:", np.ptp(matrix)) 
    print("")


Type: <class 'numpy.ndarray'>
Shape: (15707, 426)
Range: 14998

Type: <class 'numpy.ndarray'>
Shape: (15707,)
Range: 7

Type: <class 'numpy.ndarray'>
Shape: (1554, 426)
Range: 6908

Type: <class 'numpy.ndarray'>
Shape: (1554,)
Range: 7



### Pre-Process Data

In [192]:
from sklearn.decomposition import PCA

pca = PCA(n_components=32, whiten=True).fit(train_features)
train_features_compressed = pca.transform(train_features)
test_features_compressed = pca.transform(test_features)
train_features_compressed.shape

(15707, 32)

### Classifiers using scikit-learn

In [193]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

linear_model = LogisticRegression(random_state=123).fit(train_features_compressed, train_labels)
nn_model = MLPClassifier(random_state=123, hidden_layer_sizes=(100, 100, 100)).fit(train_features_compressed, train_labels)

In [207]:
print(f"Linear model Accuracy: {linear_model.score(train_features_compressed, train_labels)}(train) {linear_model.score(test_features_compressed, test_labels)}(test)")

Linear model Accuracy: 0.8553511173362195(train) 0.833976833976834(test)


In [209]:
print(f"NN model Accuracy: {nn_model.score(train_features_compressed, train_labels)}(train) {nn_model.score(test_features_compressed, test_labels)}(test)")

NN model Accuracy: 0.9998726682370918(train) 0.8371943371943372(test)


### Classifiers using PyTorch

In [196]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

In [197]:
train_dataset = TensorDataset(
    torch.tensor(train_features_compressed, dtype=torch.float),
    torch.tensor(train_labels, dtype=torch.long)
)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(
    torch.tensor(test_features_compressed, dtype=torch.float),
    torch.tensor(test_labels, dtype=torch.long)
)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [202]:
def model_accuracy(model, dataloader):
    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():
        for features, labels in dataloader:
            outputs = model(features)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [199]:
def train_model(model, dataloader):
    optim = torch.optim.SGD(model.parameters(), lr=1e-2, weight_decay=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(100):
        model.train()
        for features, labels in dataloader:
            optim.zero_grad()
            outputs = model(features)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optim.step()

        print(f"Epoch {epoch}: Accuracy = {model_accuracy(model, dataloader)}")

In [203]:
pt_linear_model = nn.Sequential(
    nn.Linear(32,8), #32 inputs, 8 outputs
)

train_model(pt_linear_model, train_dataloader)
print(f"PyTorch Linear Model accuracy on test set: {model_accuracy(pt_linear_model, test_dataloader)}")

Epoch 0: Accuracy = 0.7599159610364805
Epoch 1: Accuracy = 0.7941045393773477
Epoch 2: Accuracy = 0.8082383650601642
Epoch 3: Accuracy = 0.8171515884637423
Epoch 4: Accuracy = 0.8223085248615267
Epoch 5: Accuracy = 0.8265104730374992
Epoch 6: Accuracy = 0.829375437702935
Epoch 7: Accuracy = 0.8310944165021965
Epoch 8: Accuracy = 0.8330043929458203
Epoch 9: Accuracy = 0.833832049404724
Epoch 10: Accuracy = 0.8351053670338066
Epoch 11: Accuracy = 0.8359330234927103
Epoch 12: Accuracy = 0.8365696823072516
Epoch 13: Accuracy = 0.8383523269879671
Epoch 14: Accuracy = 0.8387979881581461
Epoch 15: Accuracy = 0.8389889858025085
Epoch 16: Accuracy = 0.8389253199210543
Epoch 17: Accuracy = 0.8395619787355956
Epoch 18: Accuracy = 0.840262303431591
Epoch 19: Accuracy = 0.8404533010759534
Epoch 20: Accuracy = 0.8405806328388616
Epoch 21: Accuracy = 0.8413446234163112
Epoch 22: Accuracy = 0.8420449481123066
Epoch 23: Accuracy = 0.8426179410453938
Epoch 24: Accuracy = 0.8430636022155726
Epoch 25: Acc

In [204]:
hidden_width = 100

# neural network with 3 hidden layers of size 100
pt_nn_model = nn.Sequential(
    nn.Linear(32, hidden_width),
    nn.ReLU(),
    nn.Linear(hidden_width, hidden_width),
    nn.ReLU(),
    nn.Linear(hidden_width, hidden_width),
    nn.ReLU(),
    nn.Linear(hidden_width, hidden_width),
    nn.ReLU(),
    nn.Linear(hidden_width, 8)
)

train_model(pt_nn_model, train_dataloader)
print(f"PyTorch NN Model accuracy on test set: {model_accuracy(pt_nn_model, test_dataloader)}")

Epoch 0: Accuracy = 0.25339020818743235
Epoch 1: Accuracy = 0.2679060291589737
Epoch 2: Accuracy = 0.3520723244413319
Epoch 3: Accuracy = 0.5359393900808557
Epoch 4: Accuracy = 0.7147131852040491
Epoch 5: Accuracy = 0.7827083465970587
Epoch 6: Accuracy = 0.812058317947412
Epoch 7: Accuracy = 0.8275291271407653
Epoch 8: Accuracy = 0.8352963646781689
Epoch 9: Accuracy = 0.8401349716686828
Epoch 10: Accuracy = 0.844209588081747
Epoch 11: Accuracy = 0.8498758515311644
Epoch 12: Accuracy = 0.8521041573820589
Epoch 13: Accuracy = 0.8572610937798434
Epoch 14: Accuracy = 0.8687209524415865
Epoch 15: Accuracy = 0.8727955688546508
Epoch 16: Accuracy = 0.875851531164449
Epoch 17: Accuracy = 0.8793531546444261
Epoch 18: Accuracy = 0.8843190933978481
Epoch 19: Accuracy = 0.883746100464761
Epoch 20: Accuracy = 0.88820271216655
Epoch 21: Accuracy = 0.8910676768319857
Epoch 22: Accuracy = 0.8940599732603298
Epoch 23: Accuracy = 0.896097281466862
Epoch 24: Accuracy = 0.9038645190042656
Epoch 25: Accura